In [ ]:
!nvidia-smi

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())
print("Total VRAM:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_fp16.eval()

print("Model loaded successfully!")
print("Model dtype:", next(model_fp16.parameters()).dtype)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")

In [ ]:
import time
import torch
import pandas as pd

def benchmark_model(model, tokenizer, prompts):
    results = []

    # Warm-up
    messages = [{"role": "user", "content": prompts[0]}]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        _ = model.generate(**encoded, max_new_tokens=20, do_sample=False)

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    total_tokens = 0
    total_time = 0

    for i, prompt in enumerate(prompts):
        messages = [{"role": "user", "content": prompt}]

        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to("cuda")

        input_length = encoded["input_ids"].shape[1]

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            output = model.generate(
                **encoded,
                max_new_tokens=128,
                do_sample=False
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        generated_tokens = output.shape[1] - input_length

        total_tokens += generated_tokens
        total_time += elapsed

        results.append({
            "Prompt": i + 1,
            "Generated Tokens": generated_tokens,
            "Time (seconds)": round(elapsed, 3),
            "TPS": round(generated_tokens / elapsed, 2)
        })

    peak_memory = torch.cuda.max_memory_allocated() / 1024**3

    return (
        pd.DataFrame(results),
        total_tokens / total_time,
        peak_memory
    )


fp16_results, fp16_tps, fp16_memory = benchmark_model(
    model_fp16, tokenizer, prompts
)

print("FP16 RESULTS")
display(fp16_results)

print("Average TPS:", round(fp16_tps, 2))
print("Peak GPU Memory:", round(fp16_memory, 3), "GB")

In [ ]:
 del model_4bit
import gc
gc.collect()
torch.cuda.empty_cache()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_fp16.eval()

fp16_responses = capture_responses(
    model_fp16, tokenizer, prompts, "Qwen2.5-0.5B-FP16"
)

display(fp16_responses[["Prompt_No", "Response"]])

fp16_responses.to_csv("fp16_responses.csv", index=False)